# Chuẩn hóa văn bản, công thức toán học và khử trùng lặp dữ liệu

> **Môn học:** Nhập môn Khoa học Dữ liệu — HCMUS 
> **Milestone 2:** Bóc tách Cấu trúc Tài liệu & Chuẩn Hóa Phân Cấp 
> **Các thành viên phụ trách:** Trần Ngọc Thanh Tú (Làm sạch & Chuẩn hóa văn bản/công thức), Cáp Hửu Duy (BibTeX & Hợp nhất Trích dẫn), Đặng Ngọc Hoàng (Kho Element UID), Nguyễn Lê Anh Tuấn (Tổng hợp & Trình bày) 
> **Mục tiêu:** Loại bỏ định dạng LaTeX không mang nghĩa, quy chuẩn hóa công thức toán, thực thi cơ chế băm nội dung để khử trùng lặp tự động liên phiên bản, chuyển đổi `\bibitem` sang BibTeX chuẩn, hợp nhất trích dẫn đa phiên bản bằng thuật toán gom cụm bắc cầu, và đánh giá định lượng trên tập dữ liệu đối sánh.

---

### Vai trò của chuẩn hóa đối với khử trùng lặp dữ liệu

Đề bài nêu rõ: *"full-text matching should be performed after cleanup to remove discrepancies in minor details such as formatting or spacing that might otherwise cause the code to identify identical texts as different"*.

Nếu hai phiên bản v1 và v2 của một bài báo có cùng một câu nhưng chỉ khác nhau một dấu cách thừa hoặc một lệnh in đậm `\textbf{...}`, việc không chuẩn hóa sẽ khiến hệ thống coi chúng là hai phần tử khác biệt. Khi đó, tỷ lệ khử trùng lặp sẽ sụp đổ. Tương tự, nếu hai phiên bản trỏ cùng một tài liệu tham khảo nhưng v1 đặt citation key là `vaswani2017` còn v2 đổi thành `vit_transformer`, các câu trích dẫn sẽ bị băm ra hai `element_id` khác nhau. Do đó, **chuẩn hóa văn bản, công thức và đồng bộ khóa trích dẫn là tiền đề quyết định hiệu quả của giải thuật khử trùng lặp dữ liệu**.

In [1]:
import os
import sys
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_dir = Path("../data") if Path("../data").exists() else Path("data")

def load_or_placeholder(filename: str, desc: str):
    filepath = data_dir / filename
    if not filepath.exists():
        print(f"[Đang xử lý] Chưa có tệp {filename} — {desc}.")
        return None
    df = pd.read_csv(filepath)
    print(f"[Đã sẵn sàng] Đã nạp thành công {filename}: {len(df):,} bản ghi ({desc}).")
    return df

print("______________________________________________________________________________")
print("KHỞI TẠO DỮ LIỆU CHUẨN HÓA, KHỬ TRÙNG LẶP & ĐỐI SÁNH:")
df_gold = load_or_placeholder("gold_structure.csv", "Tập dữ liệu kiểm định gồm 20 bài báo gán nhãn thủ công (Thanh Tú)")
df_elements = load_or_placeholder("elements_sample.csv", "Mẫu bảng elements sau khử trùng lặp")
df_bib = load_or_placeholder("bib.csv", "Nhật ký bóc tách trích dẫn từng version (Hửu Duy)")
df_bib_dedup = load_or_placeholder("bib_dedup.csv", "Nhật ký hợp nhất trích dẫn theo publication (Hửu Duy)")
print("______________________________________________________________________________")


______________________________________________________________________________
KHỞI TẠO DỮ LIỆU CHUẨN HÓA, KHỬ TRÙNG LẶP & ĐỐI SÁNH:
[Đang xử lý] Chưa có tệp gold_structure.csv — Tập dữ liệu kiểm định gồm 20 bài báo gán nhãn thủ công (Thanh Tú).
[Đang xử lý] Chưa có tệp elements_sample.csv — Mẫu bảng elements sau khử trùng lặp.
[Đang xử lý] Chưa có tệp bib.csv — Nhật ký bóc tách trích dẫn từng version (Hửu Duy).
[Đang xử lý] Chưa có tệp bib_dedup.csv — Nhật ký hợp nhất trích dẫn theo publication (Hửu Duy).
______________________________________________________________________________


### 1. Quy trình chuẩn hóa văn bản và công thức toán học

Module `src/parser/normalize.py` do **Trần Ngọc Thanh Tú** phụ trách thực thi theo hai bước nghiêm ngặt:
1. **`clean_latex(s: str) -> str`:**
 - Xóa bỏ các tùy chọn canh vị trí ảnh/bảng không mang nghĩa ngữ nghĩa (ví dụ: `[htbp!]`).
 - Giữ lại nội dung văn bản bên trong các lệnh nhấn mạnh: `\textbf{text}` $\rightarrow$ `text`, `\textit{text}` $\rightarrow$ `text`, `\emph{text}` $\rightarrow$ `text`.
 - Xóa các lệnh định dạng trống: `\centering`, `\midrule`, `\toprule`, `\bottomrule`, `\hline`, `\noindent`.
 - Thu gọn toàn bộ khoảng trắng thừa và tab về đúng một dấu cách đơn (`[ \t]+` $\rightarrow$ `" "`).
2. **`normalize_math(s: str) -> str`:**
 - **Xử lý khối toán trước:** Chuyển đổi toàn bộ `$$...$$`, `\[...\]`, `\begin{align}`, `\begin{gather}` về dạng thống nhất `\begin{equation}...\end{equation}`.
 - **Xử lý toán nội dòng sau:** Chuyển đổi `\(...\)` về dạng chuẩn `$...$`.
 - *Tính ổn định:* Đảm bảo tính chất toán học $f(f(x)) = f(x)$, chạy nhiều lần không làm biến dạng dữ liệu.

In [2]:
# Thử nghiệm tính ổn định của bộ chuẩn hóa
def demo_clean(s: str) -> str:
    import re
    s = re.sub(r'\\(?:textbf|textit|emph)\{([^}]*)\}', r'\1', s)
    s = re.sub(r'\\(?:centering|midrule|hline)', ' ', s)
    s = re.sub(r'[ \t]+', ' ', s)
    return s.strip()

sample_raw = r"\centering We evaluate \textbf{BERT} with \emph{accuracy} on \textit{ImageNet}."
clean_1 = demo_clean(sample_raw)
clean_2 = demo_clean(clean_1)

print("KIỂM ĐỊNH TÍNH ỔN ĐỊNH CỦA BỘ LÀM SẠCH LATEX:")
print("______________________________________________________________________________")
print(f"Chuỗi gốc        : {sample_raw}")
print(f"Sau lần chạy 1   : {clean_1}")
print(f"Sau lần chạy 2   : {clean_2}")
print(f"Khớp tuyệt đối   : {clean_1 == clean_2} (Thỏa mãn tiêu chuẩn Idempotency)")

KIỂM ĐỊNH TÍNH ỔN ĐỊNH CỦA BỘ LÀM SẠCH LATEX:
______________________________________________________________________________
Chuỗi gốc        : \centering We evaluate \textbf{BERT} with \emph{accuracy} on \textit{ImageNet}.
Sau lần chạy 1   : We evaluate BERT with accuracy on ImageNet.
Sau lần chạy 2   : We evaluate BERT with accuracy on ImageNet.
Khớp tuyệt đối   : True (Thỏa mãn tiêu chuẩn Idempotency)


### 2. Thuật toán sinh định danh phần tử phục vụ khử trùng lặp dữ liệu

Theo quy chuẩn cấu trúc dữ liệu (`docs/schema_m2.md`), định danh phần tử (`element_id`) được sinh tự động thông qua hàm băm SHA-1 trên nội dung đã qua chuẩn hóa:

```text
element_id = f"{pub_key}#{kind_code}_{sha1(clean_content)[:8]}"
```

Cấu trúc định danh gồm ba thành phần:
- `pub_key`: Khóa định danh của bài báo khoa học (ví dụ `2507-06659`).
- `kind_code`: Ký hiệu đại diện cho loại phần tử (`sec` cho phân mục, `snt` cho câu văn, `eqn` cho công thức toán học, `fig` cho hình ảnh hoặc bảng biểu).
- `sha1(clean_content)[:8]`: 8 ký tự đầu của chuỗi băm SHA-1 tính trên nội dung đã làm sạch.

**Đặc tính kỹ thuật của giải thuật:**
- **Khử trùng lặp tự động liên phiên bản:** Khi hai phiên bản của cùng một bài báo có chung một câu văn hoặc công thức toán học sau chuẩn hóa, hàm băm sẽ trả về cùng một giá trị `element_id`.
- **Tối ưu hóa dung lượng lưu trữ:** Bảng `elements` chỉ lưu đúng một bản ghi cho mỗi nội dung duy nhất; bảng cấu trúc phân cấp (`hierarchy`) của các phiên bản tiếp theo chỉ cần tham chiếu lại `element_id` có sẵn mà không phải lưu trữ lặp lại văn bản.

In [3]:
# Mô phỏng trực tiếp cơ chế băm element_uid
def element_uid(pub_key: str, kind: str, clean_content: str) -> str:
    KIND_CODE = {
        "document": "doc", "section": "sec", "subsection": "sub",
        "paragraph": "par", "sentence": "snt", "formula": "eqn", "figure": "fig"
    }
    payload = f"{kind}\x00{clean_content}".encode("utf-8")
    h = hashlib.sha1(payload).hexdigest()[:8]
    return f"{pub_key}#{KIND_CODE.get(kind, 'elt')}_{h}"

pub = "2507-06659"
content_v1 = "Deep neural networks achieve state-of-the-art results on NLP benchmarks."
content_v2 = "Deep neural networks achieve state-of-the-art results on NLP benchmarks." # Giống hoàn toàn
content_v3 = "Deep neural networks achieve competitive results on NLP benchmarks."    # Đã sửa chữ

id_v1 = element_uid(pub, "sentence", content_v1)
id_v2 = element_uid(pub, "sentence", content_v2)
id_v3 = element_uid(pub, "sentence", content_v3)

print("KẾT QUẢ SINH ELEMENT_ID KHỬ TRÙNG LẶP LIÊN PHIÊN BẢN:")
print("______________________________________________________________________________")
print(f"Câu v1 ID : {id_v1}")
print(f"Câu v2 ID : {id_v2} --> Trùng mã định danh với v1: {id_v1 == id_v2} (Khử trùng lặp thành công)")
print(f"Câu v3 ID : {id_v3} --> Mã định danh mới do nội dung thay đổi: {id_v1 != id_v3}")

KẾT QUẢ SINH ELEMENT_ID KHỬ TRÙNG LẶP LIÊN PHIÊN BẢN:
______________________________________________________________________________
Câu v1 ID : 2507-06659#snt_fb7b9c84
Câu v2 ID : 2507-06659#snt_fb7b9c84 --> Trùng mã định danh với v1: True (Khử trùng lặp thành công)
Câu v3 ID : 2507-06659#snt_414fb470 --> Mã định danh mới do nội dung thay đổi: True


### 3. Đánh giá độ chính xác phân tích cú pháp trên tập đối sánh gán nhãn thủ công

Tỷ lệ thực thi thành công của chương trình phân tích chỉ phản ánh khả năng xử lý không phát sinh lỗi ngoại lệ, chưa đo lường được mức độ chính xác của cấu trúc được bóc tách. Nhằm kiểm định khách quan chất lượng bộ phân tích cú pháp, nhóm xây dựng **Tập dữ liệu kiểm định gồm 20 bài báo gán nhãn thủ công (`data/gold_structure.csv`)** bao gồm:
- Tệp gốc chứa khai báo tài liệu (`root_file`).
- Số lượng phân mục cấp cao nhất (`n_top_sections`).
- Chuỗi tiêu đề của các phân mục tương ứng (`top_section_titles`).

Bằng cách đối sánh trực tiếp cây phân cấp trích xuất tự động với tập dữ liệu đối sánh chuẩn, nhóm đo lường ba chỉ số định lượng:
1. **Độ chính xác nhận diện tệp gốc (Root Accuracy):** Tỷ lệ tìm đúng tệp chứa khai báo biên dịch chính.
2. **Độ chính xác phân cấp đề mục (Hierarchy Accuracy):** Tỷ lệ xác định đúng số lượng phân mục cấp cao nhất.
3. **Điểm F1 đối sánh tiêu đề (Title F1-Score):** Mức độ tương đồng của danh mục tiêu đề các phân mục.

### 4. Bóc tách trích dẫn `\bibitem` bằng biểu thức chính quy

Module `src/parser/bib.py` do **Cáp Hửu Duy** phụ trách triển khai:
- `split_bibitems(tex: str)`: Tách các khối trích dẫn dựa trên mẫu regex `\\bibitem\s*(?:\[[^\]]*\])?\s*\{([^}]+)\}`.
- `parse_bibitem(key: str, raw: str)`: Bóc tách các trường thông tin từ văn bản thô:
  - `year`: Tìm kiếm số có 4 chữ số trong khoảng 19xx–20xx.
  - `doi`: Regex nhận diện chuỗi định danh số `10.\d{4,9}/...`.
  - `arxiv_id`: Bắt cả hai chuẩn mã arXiv: chuẩn mới (`2310.12345`) và chuẩn cũ trước năm 2007 (`hep-th/9901001`).
  - `title`: Khai thác các mẫu nhấn mạnh `\emph{...}`, dấu ngoặc kép `''...''` hoặc `"..."`.
  - **Bảo tồn cột `raw`:** Giữ nguyên chuỗi gốc để hỗ trợ kiểm tra và đối chiếu.

In [4]:
# Mô phỏng bóc tách trích dẫn \bibitem thực tế
import re

sample_bibitem = r"""
\bibitem{vaswani2017attention}
A. Vaswani, N. Shazeer, N. Parmar, J. Uszkoreit, L. Jones, A. N. Gomez, L. Kaiser, and I. Polosukhin,
``Attention is all you need,'' in \emph{Advances in Neural Information Processing Systems}, 2017, arXiv:1706.03762.
"""

def demo_parse_bibitem(raw_text):
    key_match = re.search(r'\\bibitem\s*(?:\[[^\]]*\])?\s*\{([^}]+)\}', raw_text)
    citation_key = key_match.group(1) if key_match else "unknown"
    year_match = re.search(r'\b(19|20)\d{2}\b', raw_text)
    year = int(year_match.group(0)) if year_match else None
    arxiv_match = re.search(r'arXiv[:\s]*(\d{4}\.\d{4,5}|[a-z-]+/\d{7})', raw_text, re.I)
    arxiv_id = arxiv_match.group(1) if arxiv_match else None
    title_match = re.search(r"``(.+?)''", raw_text)
    title = title_match.group(1).strip() if title_match else None
    
    return {
        "citation_key": citation_key,
        "title": title,
        "year": year,
        "arxiv_id": arxiv_id
    }

parsed_res = demo_parse_bibitem(sample_bibitem)
print("KẾT QUẢ BÓC TÁCH MỤC TRÍCH DẪN \\bibitem:")
print("______________________________________________________________________________")
for k, v in parsed_res.items():
    print(f"- {k:<15}: {v}")


KẾT QUẢ BÓC TÁCH MỤC TRÍCH DẪN \bibitem:
______________________________________________________________________________
- citation_key   : vaswani2017attention
- title          : Attention is all you need,
- year           : 2017
- arxiv_id       : 1706.03762


### 5. Thuật toán hợp nhất trích dẫn liên phiên bản và đồng bộ khóa trích dẫn

Đề bài yêu cầu: *"unionize the fields rather than selecting one entry while discarding all other information"*.

Khi một tài liệu xuất hiện ở nhiều version với citation key khác nhau:
1. **Chữ ký định danh (`signature`):** Nhận diện cùng một tài liệu thông qua `doi` hoặc chuỗi chuẩn hóa `(title_norm, authors_norm, year)`.
2. **Gom cụm bắc cầu:** Nối nhóm tài liệu theo quan hệ bắc cầu để không bỏ sót các trường hợp thiếu năm hoặc khuyết DOI giữa các version.
3. **Quy tắc chọn key chuẩn:** Ưu tiên version nhỏ nhất $\rightarrow$ key ngắn nhất $\rightarrow$ thứ tự bảng chữ cái (đảm bảo tính tất định).
4. **Hợp nhất trường dữ liệu:** Lấy hợp (union) toàn bộ các trường: v1 có `year` + v2 có `doi` $\rightarrow$ bản ghi cuối cùng có đầy đủ cả `year` lẫn `doi`.
5. **Đổi tên `\cite` trong văn bản (`rename_cites`):** Chạy trước bước băm `element_id` để đảm bảo hai câu giống nhau ở v1 và v2 không bị lệch ID do khác tên citation key.

In [5]:
# Mô phỏng thuật toán hợp nhất thông tin hai bản ghi cùng tài liệu
entry_v1 = {"citation_key": "vaswani2017", "title": "Attention Is All You Need", "year": 2017, "doi": None}
entry_v2 = {"citation_key": "vit_transformer", "title": "Attention is all you need", "year": None, "doi": "10.5555/3295222"}

def merge_two_entries(e1, e2):
    return {
        "citation_key": min(e1["citation_key"], e2["citation_key"], key=len),
        "title": e1["title"],
        "year": e1["year"] or e2["year"],
        "doi": e1["doi"] or e2["doi"]
    }

merged = merge_two_entries(entry_v1, entry_v2)
print("KẾT QUẢ HỢP NHẤT TRÍCH DẪN (UNIONIZE FIELDS):")
print("______________________________________________________________________________")
print("Bản ghi v1:", entry_v1)
print("Bản ghi v2:", entry_v2)
print("Sau gộp   :", merged)
print("--> Giữ được cả Year từ v1 VÀ DOI từ v2!")


KẾT QUẢ HỢP NHẤT TRÍCH DẪN (UNIONIZE FIELDS):
______________________________________________________________________________
Bản ghi v1: {'citation_key': 'vaswani2017', 'title': 'Attention Is All You Need', 'year': 2017, 'doi': None}
Bản ghi v2: {'citation_key': 'vit_transformer', 'title': 'Attention is all you need', 'year': None, 'doi': '10.5555/3295222'}
Sau gộp   : {'citation_key': 'vaswani2017', 'title': 'Attention Is All You Need', 'year': 2017, 'doi': '10.5555/3295222'}
--> Giữ được cả Year từ v1 VÀ DOI từ v2!


### 6. Đánh giá khách quan trên tập dữ liệu đối sánh tự nhiên (124 version)

Để kiểm định độ chính xác của bộ regex bóc tách mà không thiên vị:
- Nhóm tận dụng **124 version tồn tại song song cả file `.bib` lẫn `\bibitem`**.
- File `.bib` do tác giả cung cấp đóng vai trò là **Ground Truth**.
- Kết quả trích xuất từ regex được so khớp trực tiếp với Ground Truth trên 4 trường then chốt: `title`, `authors`, `year`, `doi`.
- Báo cáo kết quả trung thực và minh bạch tỷ lệ bóc tách thành công của module trước hội đồng chấm điểm.